# Pickle imports of top Hugging Face models

In [1]:
import asyncio
import html
import json
import os
import re
from typing import Optional
import aiohttp

In [2]:
# Number of models to consider
top_n = 10_000

In [3]:
http = aiohttp.ClientSession(connector=aiohttp.TCPConnector(limit=100, ttl_dns_cache=300))

## List top models

In [4]:
async def http_get(url: str) -> tuple[any, any]:
    retry_count = 0
    while True:
        async with http.get(url) as response:
            if response.status == 404:
                return (None, None)

            if response.status == 429:
                await asyncio.sleep(3)
                continue

            if response.status >= 500 and retry_count < 10:
                retry_count += 1
                await asyncio.sleep(3)
                continue

            if response.status >= 400:
                raise RuntimeError(f"HTTP error: {response.status}\n{await response.text()}")

            return (await response.read(), response.headers)

async def get_page(url: str) -> tuple[any, Optional[str]]:
    data, headers = await http_get(url)
    return (
        json.loads(data.decode('utf-8')) if data is not None else None,
        headers['Link'].split(';')[0].strip('<>') if headers is not None and 'Link' in headers else None)
        
async def get_top_models(model_count: int) -> list[dict]:
    models = []
    page_url = 'https://huggingface.co/api/models?sort=downloads&direction=-1&limit=100'
    while len(models) < model_count:
        page_models, page_url = await get_page(page_url)
        models.extend(page_models)
        if page_url is None or len(models) >= model_count:
            return models[:model_count]

In [5]:
models = await get_top_models(top_n)

In [6]:
os.makedirs('data', exist_ok=True)

with open('data/models.jsonl', 'w') as f:
    for model in models:
        f.write(json.dumps(model) + '\n')

## List model files with Pickle imports

In [7]:
with open('data/models.jsonl', 'r') as f:
    model_ids = [json.loads(line)['id'] for line in f]

assert len(model_ids) == top_n, f"Expected: {top_n}, actual: {len(model_ids)}"

In [8]:
async def get_model_info(model_id: str) -> dict:
    # Get the model info by downloading the model HTML page and extracting the data-props attribute
    # (there is probably a better way to do this via some REST API but I have not found it)
    data, _ = await http_get(f"https://huggingface.co/{model_id}/tree/main")
    model_info_line = next(line for line in data.decode('utf-8').split("\n") if 'data-target="ViewerIndexTreeList"' in line)
    return json.loads(html.unescape(re.search(r'data-props="([^"]+)"', model_info_line).group(1)))

async def get_model_security_info(model_id: str) -> list[dict]:
    model_info = await get_model_info(model_id)
    model_id = model_info["context"]["repo"]["name"];
    return [{
        "model_id": model_id,
        "path": item["path"],
        "pickleSafetyLevel": item["security"]["pickleImportScan"]["highestSafetyLevel"],
        "pickleImports": item["security"]["pickleImportScan"]["imports"],
    } for item in model_info["entries"] if item["type"] == "file" and item.get("security") is not None and item["security"]["pickleImportScan"] is not None]

In [9]:
model_files = []
for i, model_id in enumerate(model_ids):
    try:
        print(f"Querying model {i+1}/{len(model_ids)} {model_id}                                                                ", end="\r")
        model_files.extend(await get_model_security_info(model_id))
    except Exception as e:
        print(f"Error while processing {model_id}: {e}")

Error while processing michellejieli/NSFW_text_classifier: coroutine raised StopIteration                                                                        
Error while processing Dremmar/nsfw-xl: coroutine raised StopIteration                                                                                                  
Error while processing enhanceaiteam/Flux-uncensored: coroutine raised StopIteration                                                                                 
Error while processing UnfilteredAI/NSFW-gen-v2: coroutine raised StopIteration                                                                                          
Error while processing stablediffusionapi/duchaiten-real3d-nsfw-xl: coroutine raised StopIteration                                                               
Error while processing stablediffusionapi/explicit-freedom-nsfw-wai: coroutine raised StopIteration                                                         
Error while pr

In [10]:
with open('data/model_files.jsonl', 'w') as f:
    for model_file in model_files:
        f.write(json.dumps(model_file) + '\n')

## List top Pickle imports

In [11]:
with open('data/model_files.jsonl', 'r') as f:
    model_files = [json.loads(line) for line in f]

In [13]:
pickle_import_stats = {}
for model_file in model_files:
    for pickle_import in model_file.get('pickleImports', []):
        pickle_import_id = pickle_import['module'] + '|' + pickle_import['name']
        stats = pickle_import_stats.get(pickle_import_id, {'count': 0, 'safety': pickle_import['safety']})
        stats['count'] += 1
        pickle_import_stats[pickle_import_id] = stats

print(f"Found {len(pickle_import_stats)} unique pickle imports")

Found 264 unique pickle imports


In [14]:
with open('data/pickle_import_stats.json', 'w') as f:
    json.dump(pickle_import_stats, f)

In [15]:
for row in sorted(pickle_import_stats.items(), key=lambda x: x[1]['count'], reverse=True):
    if row[1]['count'] > 10:
        print(row)

('collections|OrderedDict', {'count': 7552, 'safety': 'innocuous'})
('torch._utils|_rebuild_tensor_v2', {'count': 7550, 'safety': 'innocuous'})
('torch|FloatStorage', {'count': 4692, 'safety': 'innocuous'})
('torch|HalfStorage', {'count': 1881, 'safety': 'innocuous'})
('torch|BFloat16Storage', {'count': 1576, 'safety': 'innocuous'})
('torch|LongStorage', {'count': 1344, 'safety': 'innocuous'})
('torch|device', {'count': 659, 'safety': 'suspicious'})
('transformers.trainer_utils|SchedulerType', {'count': 610, 'safety': 'suspicious'})
('transformers.trainer_utils|IntervalStrategy', {'count': 599, 'safety': 'suspicious'})
('transformers.trainer_utils|HubStrategy', {'count': 553, 'safety': 'suspicious'})
('transformers.training_args|OptimizerNames', {'count': 521, 'safety': 'suspicious'})
('transformers.training_args|TrainingArguments', {'count': 416, 'safety': 'suspicious'})
('torch|ByteStorage', {'count': 406, 'safety': 'innocuous'})
('accelerate.utils.dataclasses|DistributedType', {'cou